# MLB Pitcher Pipeline
1. Drive zip 파일 → 화면전환 감지 → 투구장면 필터링 → JSON 저장
2. 투구장면 → 투수선택(y2최대) → 릴리스프레임(x변위×속도) → 좌표 추출 → CSV 저장

In [1]:
# ── 0. 패키지 설치 & 드라이브 마운트 ─────────────────────────────
import subprocess, sys, os
def pip(*args): subprocess.check_call([sys.executable, "-m", "pip", *args])

for pkg in ["scenedetect[opencv]", "ultralytics", "mediapipe", "pyarrow", "scipy"]:
    print(f"  {pkg} ...", end=" ", flush=True)
    pip("install", "-q", pkg)
    print("OK")

from google.colab import drive
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
print("드라이브 마운트 완료")

  scenedetect[opencv] ... OK
  ultralytics ... OK
  mediapipe ... OK
  pyarrow ... OK
  scipy ... OK
Mounted at /content/drive
드라이브 마운트 완료


In [2]:
# ── 1. 설정 ───────────────────────────────────────────────────────
TARGET_SLOT = 1   # ← 0,1,2 중 선택 (3=2024, 4=2025 영상은 아직 Drive에 없음)

ZIP_DIR    = "/content/drive/MyDrive/MLB_pitcher/video_zips"
OUTPUT_DIR = "/content/drive/MyDrive/MLB_pitcher/output"
WORK_DIR   = "/tmp/mlb_work"

# play_id↔game_pk 매핑 + 손잡이 소스: 06이 만든 sample(경기당 15개, game_pk 100% 조인)
PLAY_IDS_SAMPLE       = "/content/drive/MyDrive/MLB_pitcher/data/play_ids_sample.csv"
STATCAST_PARQUET_GLOB = "/content/drive/MyDrive/MLB_pitcher/statcast/statcast_*.parquet"

MAX_GAMES_PER_PITCHER_SEASON = 3  # 투수당 시즌당 최대 경기 수

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(WORK_DIR,   exist_ok=True)
print(f"ZIP_DIR        : {ZIP_DIR}")
print(f"OUTPUT_DIR     : {OUTPUT_DIR}")
print(f"PLAY_IDS_SAMPLE: {PLAY_IDS_SAMPLE}")
print(f"TARGET_SLOT    : {TARGET_SLOT}")

ZIP_DIR        : /content/drive/MyDrive/MLB_pitcher/video_zips
OUTPUT_DIR     : /content/drive/MyDrive/MLB_pitcher/output
PLAY_IDS_SAMPLE: /content/drive/MyDrive/MLB_pitcher/data/play_ids_sample.csv
TARGET_SLOT    : 1


In [ ]:
# ── 2. 상수 & 모델 로드 ───────────────────────────────────────────
import cv2, numpy as np, mediapipe as mp, pandas as pd, glob as _glob, os, urllib.request, torch
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision
from ultralytics import YOLO
from scipy.signal import savgol_filter

# ── 투구장면 판별 기준 ──────────────────────────────────────────
N_MIN, MB_MAX, GR_MIN, GR_MAX = 4, 0.15, 0.15, 0.45
CUT_THRESHOLD = 27.0

# ── 관절 정의 ───────────────────────────────────────────────────
JOINT_NAMES = [
    "left_ear","right_ear",
    "left_shoulder","right_shoulder",
    "left_elbow","right_elbow",
    "left_wrist","right_wrist",
    "left_hip","right_hip",
    "left_knee","right_knee",
    "left_ankle","right_ankle",
]
MP_IDX  = [7,8,11,12,13,14,15,16,23,24,25,26,27,28]
J       = len(JOINT_NAMES)
NAME2I  = {n:i for i,n in enumerate(JOINT_NAMES)}

PAD          = 25
CONF_THR     = 0.3
SG_WINDOW    = 7
SG_POLYORDER = 2

# 전신 판단 기준: 양어깨, 양힙, 양무릎, 양발목 모두 보여야 투수 후보
FULL_BODY_KPS = [5, 6, 11, 12, 13, 14, 15, 16]

# ── GPU 사용 여부 ───────────────────────────────────────────────
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"DEVICE: {DEVICE}")
if DEVICE == "cpu":
    print("  ⚠ GPU 미사용. [런타임 → 런타임 유형 변경 → T4 GPU] 후 처음부터 재실행 권장")

# ── Pose Landmarker 모델 (Tasks API) ────────────────────────────
MODEL_PATH = "pose_landmarker.task"
if not os.path.exists(MODEL_PATH):
    print("Pose 모델 다운로드...", end=" ", flush=True)
    urllib.request.urlretrieve(
        "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_full/float16/latest/pose_landmarker_full.task",
        MODEL_PATH
    )
    print("완료")

def make_landmarker():
    opts = mp_vision.PoseLandmarkerOptions(
        base_options=mp_python.BaseOptions(model_asset_path=MODEL_PATH),
        num_poses=1,
        min_pose_detection_confidence=0.5,
        min_pose_presence_confidence=0.5,
        min_tracking_confidence=0.5,
    )
    return mp_vision.PoseLandmarker.create_from_options(opts)

LANDMARKER = make_landmarker()

# ── YOLO Pose (투수 선택용) ──────────────────────────────────────
det_model = YOLO("yolov8n-pose.pt")
det_model.to(DEVICE)
print(f"mediapipe {mp.__version__} | YOLO Pose 로드 완료 (device={DEVICE})")

# ── play_ids_sample 로드 (play_id, pitcher_id, game_pk, season) ──
print("샘플 데이터 로드...", end=" ", flush=True)
_sample_df = pd.read_csv(PLAY_IDS_

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
DEVICE: cuda
Pose 모델 다운로드... 완료
mediapipe 0.10.35 | YOLO Pose 로드 완료 (device=cuda)
샘플 데이터 로드... 완료 (93,466개, p_throws 결측 0.0%)
경기 수 필터 빌드... 완료 (63,128개 허용)


In [ ]:
SAMPLE, dtype=str)
_sample_df["game_pk"] = _sample_df["game_pk"].astype(int)

_throws_map = {}
for _p in _glob.glob(STATCAST_PARQUET_GLOB):
    _df = pd.read_parquet(_p, columns=["pitcher", "p_throws"]).dropna(subset=["pitcher", "p_throws"])
    _throws_map.update(dict(zip(_df["pitcher"].astype(int).astype(str), _df["p_throws"])))
_sample_df["p_throws"] = _sample_df["pitcher_id"].map(_throws_map)
PLAY_THROWS = dict(zip(_sample_df["play_id"], _sample_df["p_throws"]))
print(f"완료 ({len(PLAY_THROWS):,}개, p_throws 결측 {_sample_df['p_throws'].isna().mean():.1%})")

# ── 투수당 시즌당 N경기 필터 ─────────────────────────────────────
print("경기 수 필터 빌드...", end=" ", flush=True)
_top_games = (
    _sample_df.groupby(["pitcher_id", "season"])["game_pk"]
    .apply(lambda x: set(sorted(x.unique())[:MAX_GAMES_PER_PITCHER_SEASON]))
)
_sample_df = _sample_df.join(_top_games.rename("allowed"), on=["pitcher_id", "season"])
PLAY_ALLOWED = set(
    _sample_df[_sample_df.apply(lambda r: r["game_pk"] in r["allowed"], axis=1)]["play_id"]
)
print(f"완료 ({len(PLAY_ALLOWED):,}개 허용)")

In [4]:
# ── 3. 헬퍼 함수 ──────────────────────────────────────────────────
from scenedetect import open_video, SceneManager
from scenedetect.detectors import ContentDetector

# ── 3-1. 투구장면 판별 ──────────────────────────────────────────
def classify_scene(video_path, start_frame, end_frame):
    cap = cv2.VideoCapture(video_path)
    fw  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    fh  = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    frame_area = fw * fh
    total = max(end_frame - start_frame, 1)

    sample_frames = []
    for pos in [0.1, 0.5, 0.9]:
        idx = start_frame + int(total * pos)
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        for _ in range(5):
            ret, f = cap.read()
            if ret:
                sample_frames.append(f)
    cap.release()

    max_ratios, green_ratios, n_persons = [], [], []
    for frame in sample_frames:
        res   = det_model(frame, classes=[0], device=DEVICE, verbose=False)[0]
        boxes = res.boxes.xyxy.cpu().numpy() if len(res.boxes) else []
        n_persons.append(len(boxes))
        if len(boxes):
            areas = [(x2-x1)*(y2-y1) for x1,y1,x2,y2 in boxes]
            max_ratios.append(max(areas) / frame_area)
        else:
            max_ratios.append(0.0)
        hsv   = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
        green = cv2.inRange(hsv, (35,40,40), (85,255,255))
        green_ratios.append(float(green.sum()) / 255.0 / frame_area)

    n  = float(np.mean(n_persons))
    mb = float(np.mean(max_ratios))
    gr = float(np.mean(green_ratios))
    is_pitching = (n >= N_MIN) and (mb < MB_MAX) and (GR_MIN < gr < GR_MAX)
    return {"is_pitching": is_pitching, "n_persons": round(n,1),
            "max_bbox_ratio": round(mb,3), "green_ratio": round(gr,3)}


def detect_pitching_scene(video_path):
    video = open_video(video_path)
    sm    = SceneManager()
    sm.add_detector(ContentDetector(threshold=CUT_THRESHOLD))
    sm.detect_scenes(video, show_progress=False)
    scenes = sm.get_scene_list()

    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps   = cap.get(cv2.CAP_PROP_FPS)
    cap.release()

    raw_cuts = [(s.frame_num, e.frame_num) for s,e in scenes] if scenes else [(0, total)]

    cuts = []
    for i, (sf, ef) in enumerate(raw_cuts):
        info = classify_scene(video_path, sf, ef)
        info.update({"scene": i+1, "start_frame": sf, "end_frame": ef,
                     "duration_sec": round((ef-sf)/fps, 2)})
        cuts.append(info)

    pitching = [c for c in cuts if c["is_pitching"]]
    if pitching:
        best = max(pitching, key=lambda c: c["end_frame"]-c["start_frame"])
    else:
        best = dict(max(cuts, key=lambda c: c["end_frame"]-c["start_frame"]), fallback=True)
    return {"total_frames": total, "fps": round(fps,2), "cuts": cuts, "pitching": best}


# ── 3-2. 포즈 유틸 ──────────────────────────────────────────────
def interpolate_bbox(bb):
    T, idx, out = len(bb), np.arange(len(bb)), bb.copy()
    for k in range(4):
        col = out[:,k]; nan = np.isnan(col)
        if nan.all(): continue
        if nan.any(): col[nan] = np.interp(idx[nan], idx[~nan], col[~nan])
        out[:,k] = col
    return out

def smooth_series(series, conf):
    T = len(series)
    s = series.astype(float).copy()
    s[conf < CONF_THR] = np.nan
    nans = np.isnan(s)
    if nans.any() and not nans.all():
        idx = np.arange(T)
        s   = np.interp(idx, idx[~nans], s[~nans])
    elif nans.all():
        return series.astype(float)
    win = min(SG_WINDOW, T-1 if (T-1)%2==1 else T-2)
    win = max(win, SG_POLYORDER+2)
    if win%2==0: win-=1
    return savgol_filter(s, win, SG_POLYORDER) if (win>=3 and T>win) else s

def smooth_coords(raw, vis):
    out = raw.astype(float).copy()
    for j in range(J):
        out[:,j,0] = smooth_series(raw[:,j,0], vis[:,j])
        out[:,j,1] = smooth_series(raw[:,j,1], vis[:,j])
    return out


# ── 3-3. 릴리스 프레임 탐지 ─────────────────────────────────────
def find_release_frame(smoothed, W, hand=None):
    hand_source = "statcast"
    if hand not in ("R", "L"):
        hand_source = "heuristic"
        rw_x = smoothed[:, NAME2I["right_wrist"], 0]
        lw_x = smoothed[:, NAME2I["left_wrist"],  0]
        rw_dist = rw_x.max() - W / 2
        lw_dist = W / 2 - lw_x.min()
        hand = "L" if rw_dist >= lw_dist else "R"

    if hand == "R":
        wrist_x    = smoothed[:, NAME2I["right_wrist"],    0]
        wrist_y    = smoothed[:, NAME2I["right_wrist"],    1]
        shoulder_x = smoothed[:, NAME2I["right_shoulder"], 0]
        arm_ext    = wrist_x - shoulder_x
    else:
        wrist_x    = smoothed[:, NAME2I["left_wrist"],     0]
        wrist_y    = smoothed[:, NAME2I["left_wrist"],     1]
        shoulder_x = smoothed[:, NAME2I["left_shoulder"],  0]
        arm_ext    = shoulder_x - wrist_x

    dx       = np.diff(wrist_x, prepend=wrist_x[0])
    dy       = np.diff(wrist_y, prepend=wrist_y[0])
    velocity = np.sqrt(dx**2 + dy**2)

    T         = len(arm_ext)
    candidate = int(np.argmax(arm_ext * velocity))
    lo = max(0, candidate - 10)
    hi = min(T, candidate + 11)
    release_frame = lo + int(np.argmax(arm_ext[lo:hi]))

    return release_frame, hand, hand_source


# ── 3-4. 투수 선택 + 포즈 추출 ──────────────────────────────────
def select_pitcher(res, H):
    """YOLO Pose 결과에서 투수 bbox 반환.
    전신 키포인트(양어깨·양힙·양무릎·양발목) 모두 보이는 사람 중 y2 최대.
    전신 조건을 만족하는 사람이 없으면 감지된 전체 중 y2 최대.
    """
    if res.boxes is None or len(res.boxes) == 0:
        return np.full(4, np.nan)

    xyxy    = res.boxes.xyxy.cpu().numpy()
    kps_all = res.keypoints

    full = []
    if kps_all is not None:
        for i in range(len(xyxy)):
            if i < len(kps_all):
                kpconf = kps_all[i].conf[0].cpu().numpy()
                if all(kpconf[j] > CONF_THR for j in FULL_BODY_KPS):
                    full.append(i)

    pool = full if full else list(range(len(xyxy)))
    idx  = pool[int(np.argmax(xyxy[pool, 3]))]
    return xyxy[idx]


def extract_pose_and_release(video_path, start_frame, end_frame, hand=None):
    cap = cv2.VideoCapture(video_path)
    W   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

    landmarker = LANDMARKER

    bb_list, raw_list, vis_list = [], [], []
    prev_raw = np.zeros((J, 2))

    for _ in range(end_frame - start_frame):
        ret, frame = cap.read()
        if not ret:
            break

        res = det_model(frame, classes=[0], device=DEVICE, verbose=False)[0]
        bb  = select_pitcher(res, H)
        bb_list.append(bb)

        if not np.isnan(bb).any():
            x1=max(0,int(bb[0])-PAD); y1=max(0,int(bb[1])-PAD)
            x2=min(W,int(bb[2])+PAD); y2=min(H,int(bb[3])+PAD)
            crop = frame[y1:y2, x1:x2]; cw,ch,ox,oy = x2-x1,y2-y1,x1,y1
        else:
            crop = frame; cw,ch,ox,oy = W,H,0,0

        raw_t = np.zeros((J,2)); vis_t = np.zeros(J)
        mp_img = mp.Image(image_format=mp.ImageFormat.SRGB,
                          data=cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
        result = landmarker.detect(mp_img)
        if result.pose_landmarks:
            lm = result.pose_landmarks[0]
            for j, mp_j in enumerate(MP_IDX):
                raw_t[j, 0] = lm[mp_j].x * cw + ox
                raw_t[j, 1] = lm[mp_j].y * ch + oy
                v = lm[mp_j].visibility
                vis_t[j] = v if v is not None else 1.0
            prev_raw = raw_t.copy()
        else:
            raw_t = prev_raw.copy()

        raw_list.append(raw_t); vis_list.append(vis_t)

    cap.release()

    if not raw_list:
        return None

    raw      = np.array(raw_list)
    vis      = np.array(vis_list)
    smoothed = smooth_coords(raw, vis)

    rel_idx, hand_out, hand_source = find_release_frame(smoothed, W, hand=hand)

    row = {
        "release_frame": start_frame + rel_idx,
        "hand":          hand_out,
        "hand_source":   hand_source,
    }
    for j, name in enumerate(JOINT_NAMES):
        row[f"{name}_x"] = round(float(raw[rel_idx, j, 0]), 2)
        row[f"{name}_y"] = round(float(raw[rel_idx, j, 1]), 2)
    return row

print("헬퍼 함수 정의 완료")

헬퍼 함수 정의 완료


/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:294: SyntaxWarning: invalid escape sequence '\d'
  lines_video = [l for l in lines if ' Video: ' in l and re.search('\d+x\d+', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:367: SyntaxWarning: invalid escape sequence '\d'
  rotation_lines = [l for l in lines if 'rotate          :' in l and re.search('\d+$', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:370: SyntaxWarning: invalid escape sequence '\d'
  match = re.search('\d+$', rotation_line)
/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"


In [5]:
# ── 4. 메인 파이프라인 (이어서 실행 지원) ─────────────────────────
import glob, zipfile, shutil, json, csv
from pathlib import Path

zip_files = sorted(glob.glob(f"{ZIP_DIR}/batch_slot{TARGET_SLOT}_*.zip"))
print(f"처리할 zip: {len(zip_files)}개")
if not zip_files:
    print("⚠ zip 파일을 찾지 못했습니다. ZIP_DIR 경로를 확인하세요.")

CSV_COLS = (["video_name", "release_frame", "hand", "hand_source"]
            + [f"{n}_{ax}" for n in JOINT_NAMES for ax in ["x","y"]])

def save_csv(path, rows):
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=CSV_COLS, extrasaction="ignore")
        w.writeheader()
        w.writerows(rows)

def save_json(path, data):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

for zip_path in zip_files:
    zip_name    = Path(zip_path).stem
    extract_dir = f"{WORK_DIR}/{zip_name}"
    json_path   = f"{OUTPUT_DIR}/{zip_name}_cuts.json"
    csv_path    = f"{OUTPUT_DIR}/{zip_name}_coords.csv"

    print(f"\n{'='*60}")
    print(f"  {zip_name}")
    print(f"{'='*60}")

    # ── 이전 진행 상황 로드 ─────────────────────────────────────
    done_set    = set()
    coords_rows = []
    cuts_result = {}

    if os.path.exists(csv_path):
        try:
            existing = pd.read_csv(csv_path)
            done_set    = set(existing["video_name"].tolist())
            coords_rows = existing.to_dict("records")
            print(f"  ▶ 이전 CSV 발견: {len(done_set)}개 완료, 이어서 진행")
        except Exception:
            pass

    if os.path.exists(json_path):
        try:
            with open(json_path, "r", encoding="utf-8") as f:
                cuts_result = json.load(f)
            print(f"  ▶ 이전 JSON 발견: {len(cuts_result)}개 씬 데이터 로드")
        except Exception:
            pass

    # ── zip 압축 해제 (이미 풀려있으면 스킵) ────────────────────
    if not os.path.exists(extract_dir):
        os.makedirs(extract_dir, exist_ok=True)
        with zipfile.ZipFile(zip_path, "r") as z:
            z.extractall(extract_dir)
        print(f"  압축 해제 완료")
    else:
        print(f"  압축 해제 디렉토리 이미 존재, 스킵")

    videos    = sorted(Path(extract_dir).rglob("*.mp4"))
    allowed   = [vp for vp in videos if vp.stem in PLAY_ALLOWED]
    remaining = [vp for vp in allowed if vp.stem not in done_set]
    print(f"  전체 {len(videos)}개 / 필터 후 {len(allowed)}개 / 미처리 {len(remaining)}개")

    for vp in remaining:
        print(f"  [{vp.name}] ", end="", flush=True)
        try:
            # ── 씬 감지 + 투구장면 필터링 ────────────────────
            scene_info = detect_pitching_scene(str(vp))
            cuts_result[vp.name] = scene_info
            save_json(json_path, cuts_result)

            pitching = scene_info.get("pitching")
            if pitching is None:
                print("투구장면 없음")
                continue

            sf, ef = pitching["start_frame"], pitching["end_frame"]
            print(f"scene={pitching['scene']} ({sf}-{ef}) ", end="", flush=True)

            # ── statcast 손잡이 조회 + 포즈 추출 ─────────────
            hand = PLAY_THROWS.get(vp.stem)
            row  = extract_pose_and_release(str(vp), sf, ef, hand=hand)
            if row is None:
                print("포즈 추출 실패")
                continue

            row["video_name"] = vp.stem
            coords_rows.append(row)
            done_set.add(vp.stem)
            save_csv(csv_path, coords_rows)
            src = row["hand_source"]
            print(f"→ release={row['release_frame']} hand={row['hand']}[{src}]")

        except Exception as e:
            print(f"오류: {e}")

    print(f"\n  완료: {len(coords_rows)}행")
    print(f"  JSON → {json_path}")
    print(f"  CSV  → {csv_path}")

    shutil.rmtree(extract_dir)

print("\n모든 zip 처리 완료")

처리할 zip: 97개

  batch_slot1_0001
  ▶ 이전 CSV 발견: 140개 완료, 이어서 진행
  ▶ 이전 JSON 발견: 140개 씬 데이터 로드
  압축 해제 완료
  전체 200개 / 필터 후 140개 / 미처리 0개

  완료: 140행
  JSON → /content/drive/MyDrive/MLB_pitcher/output/batch_slot1_0001_cuts.json
  CSV  → /content/drive/MyDrive/MLB_pitcher/output/batch_slot1_0001_coords.csv

  batch_slot1_0002
  ▶ 이전 CSV 발견: 135개 완료, 이어서 진행
  ▶ 이전 JSON 발견: 135개 씬 데이터 로드
  압축 해제 완료
  전체 200개 / 필터 후 135개 / 미처리 0개

  완료: 135행
  JSON → /content/drive/MyDrive/MLB_pitcher/output/batch_slot1_0002_cuts.json
  CSV  → /content/drive/MyDrive/MLB_pitcher/output/batch_slot1_0002_coords.csv

  batch_slot1_0003
  ▶ 이전 CSV 발견: 115개 완료, 이어서 진행
  ▶ 이전 JSON 발견: 115개 씬 데이터 로드
  압축 해제 완료
  전체 200개 / 필터 후 115개 / 미처리 0개

  완료: 115행
  JSON → /content/drive/MyDrive/MLB_pitcher/output/batch_slot1_0003_cuts.json
  CSV  → /content/drive/MyDrive/MLB_pitcher/output/batch_slot1_0003_coords.csv

  batch_slot1_0004
  ▶ 이전 CSV 발견: 125개 완료, 이어서 진행
  ▶ 이전 JSON 발견: 125개 씬 데이터 로드
  압축 해제 완료
  전체 199개 / 필터 후 12

INFO:pyscenedetect:Detecting scenes...


scene=1 (0-415) → release=186 hand=R[statcast]
  [7e14e8ae-e060-458a-9374-bfa1b6a889b3.mp4] 

INFO:pyscenedetect:Detecting scenes...


scene=1 (0-261) → release=183 hand=R[statcast]
  [7e76ed28-8302-4197-a9a3-15b88313192e.mp4] 

...

scene=1 (0-464) → release=182 hand=R[statcast]
  [f91c8268-1240-4a6e-a00b-98745fcf1da0.mp4] 

INFO:pyscenedetect:Detecting scenes...


scene=1 (0-252) → release=181 hand=R[statcast]
  [fdbee5d4-ec34-4b05-a8d1-10fab11e5329.mp4] 

INFO:pyscenedetect:Detecting scenes...


scene=2 (118-438) → release=185 hand=R[statcast]

  완료: 39행
  JSON → /content/drive/MyDrive/MLB_pitcher/output/batch_slot1_0096_cuts.json
  CSV  → /content/drive/MyDrive/MLB_pitcher/output/batch_slot1_0096_coords.csv

  batch_slot1_0097
  압축 해제 완료
  전체 3개 / 필터 후 0개 / 미처리 0개

  완료: 0행
  JSON → /content/drive/MyDrive/MLB_pitcher/output/batch_slot1_0097_cuts.json
  CSV  → /content/drive/MyDrive/MLB_pitcher/output/batch_slot1_0097_coords.csv

모든 zip 처리 완료
